# ACN Dataset Preprocessing

In [3]:
import pandas as pd
import numpy as np
import json

In [4]:
with open(r"F:\EV_TARIFF_OPTIMIZATION\data\acn_data.json", "r") as f:
    data = json.load(f)

print(type(data))

<class 'dict'>


In [5]:
data.keys()

dict_keys(['_meta', '_items'])

In [6]:
df = pd.json_normalize(data['_items'])

In [7]:
df.head()

,_id,clusterID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,5e210e1ff9af8b57bb4f5500,0039,"Wed, 01 Jan 2020 02:12:11 GMT","Thu, 02 Jan 2020 00:05:40 GMT","Wed, 01 Jan 2020 06:55:27 GMT",15.813,2_39_139_28_2020-01-01 02:12:11.015844,0002,CA-303,2-39-139-28,America/Los_Angeles,000000838,"[{'WhPerMile': 600, 'kWhRequested': 36.0, 'mil..."
1,5e225f9ff9af8b5c26d21715,0039,"Wed, 01 Jan 2020 09:42:14 GMT","Thu, 02 Jan 2020 02:25:40 GMT","Wed, 01 Jan 2020 14:42:11 GMT",32.020,2_39_131_30_2020-01-01 09:42:14.259248,0002,CA-305,2-39-131-30,America/Los_Angeles,000000067,"[{'WhPerMile': 250, 'kWhRequested': 60.0, 'mil..."
2,5e225f9ff9af8b5c26d21716,0039,"Wed, 01 Jan 2020 18:10:34 GMT","Wed, 01 Jan 2020 21:05:40 GMT","Wed, 01 Jan 2020 19:22:56 GMT",2.328,2_39_127_19_2020-01-01 18:10:34.057445,0002,CA-309,2-39-127-19,America/Los_Angeles,000000710,"[{'WhPerMile': 261, 'kWhRequested': 7.83, 'mil..."
3,5e225f9ff9af8b5c26d21717,0039,"Wed, 01 Jan 2020 19:44:51 GMT","Thu, 02 Jan 2020 01:23:37 GMT","Wed, 01 Jan 2020 22:43:57 GMT",19.868,2_39_79_377_2020-01-01 19:44:51.127414,0002,CA-325,2-39-79-377,America/Los_Angeles,000000248,"[{'WhPerMile': 250, 'kWhRequested': 20.0, 'mil..."
4,5e225f9ff9af8b5c26d21718,0039,"Thu, 02 Jan 2020 01:12:29 GMT","Thu, 02 Jan 2020 04:38:39 GMT","Thu, 02 Jan 2020 03:11:48 GMT",8.336,2_39_126_20_2020-01-02 01:12:28.778216,0002,CA-310,2-39-126-20,America/Los_Angeles,000001099,"[{'WhPerMile': 400, 'kWhRequested': 24.0, 'mil..."


In [8]:
df.shape

(2050, 13)

In [10]:
df.columns

Index(['_id', 'clusterID', 'connectionTime', 'disconnectTime',
       'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID',
       'stationID', 'timezone', 'userID', 'userInputs'],
      dtype='object')

In [7]:
df.to_csv(r"F:\EV_Tariff_Optimization\data\acn_data.csv", index=False)

## Feature Engineering and Data Cleaning

In [10]:
df['connectionTime'] = pd.to_datetime(df['connectionTime'])
df['disconnectTime'] = pd.to_datetime(df['disconnectTime'])
df['doneChargingTime'] = pd.to_datetime(df['doneChargingTime'])

In [11]:
df['session_duration_hr'] = (
    df['disconnectTime'] - df['connectionTime']
).dt.total_seconds() / 3600

df['charging_duration_hr'] = (
    df['doneChargingTime'] - df['connectionTime']
).dt.total_seconds() / 3600

df['hour'] = df['connectionTime'].dt.hour
df['day_of_week'] = df['connectionTime'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

df['fixed_tariff'] = 15
df['fixed_revenue'] = df['kWhDelivered'] * df['fixed_tariff']

In [12]:
df_clean = df[
    (df['kWhDelivered'] > 0) &
    (df['session_duration_hr'] > 0) &
    (df['charging_duration_hr'] > 0)
].copy()

df_clean.shape

(1782, 20)

## Save Cleaned Dataset

In [13]:
df_clean.to_csv(r"F:\EV_Tariff_Optimization\data\cleaned_acn_data.csv", index=False)

In [17]:
df_clean.head()
#df_clean.shape

,_id,clusterID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs,session_duration_hr,charging_duration_hr,hour,day_of_week,is_weekend,fixed_tariff,fixed_revenue
0,5e210e1ff9af8b57bb4f5500,0039,2020-01-01 02:12:11,2020-01-02 00:05:40,2020-01-01 06:55:27,15.813,2_39_139_28_2020-01-01 02:12:11.015844,0002,CA-303,2-39-139-28,America/Los_Angeles,000000838,"[{'WhPerMile': 600, 'kWhRequested': 36.0, 'mil...",21.891389,4.721111,2,Wednesday,0,15,237.195
1,5e225f9ff9af8b5c26d21715,0039,2020-01-01 09:42:14,2020-01-02 02:25:40,2020-01-01 14:42:11,32.020,2_39_131_30_2020-01-01 09:42:14.259248,0002,CA-305,2-39-131-30,America/Los_Angeles,000000067,"[{'WhPerMile': 250, 'kWhRequested': 60.0, 'mil...",16.723889,4.999167,9,Wednesday,0,15,480.300
2,5e225f9ff9af8b5c26d21716,0039,2020-01-01 18:10:34,2020-01-01 21:05:40,2020-01-01 19:22:56,2.328,2_39_127_19_2020-01-01 18:10:34.057445,0002,CA-309,2-39-127-19,America/Los_Angeles,000000710,"[{'WhPerMile': 261, 'kWhRequested': 7.83, 'mil...",2.918333,1.206111,18,Wednesday,0,15,34.920
3,5e225f9ff9af8b5c26d21717,0039,2020-01-01 19:44:51,2020-01-02 01:23:37,2020-01-01 22:43:57,19.868,2_39_79_377_2020-01-01 19:44:51.127414,0002,CA-325,2-39-79-377,America/Los_Angeles,000000248,"[{'WhPerMile': 250, 'kWhRequested': 20.0, 'mil...",5.646111,2.985000,19,Wednesday,0,15,298.020
4,5e225f9ff9af8b5c26d21718,0039,2020-01-02 01:12:29,2020-01-02 04:38:39,2020-01-02 03:11:48,8.336,2_39_126_20_2020-01-02 01:12:28.778216,0002,CA-310,2-39-126-20,America/Los_Angeles,000001099,"[{'WhPerMile': 400, 'kWhRequested': 24.0, 'mil...",3.436111,1.988611,1,Thursday,0,15,125.040
